In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import folium

In [4]:
gdf_centroids_final = pd.read_csv("gdf_centroids_final.csv")
snap_atlanta_df = pd.read_csv("snap_atlanta_df.csv")
df_grocery_selected = pd.read_csv("df_grocery_selected.csv")
gdf_selected = pd.read_csv("gdf_selected.csv")


In [5]:
gdf_centroids_final

,LIVUNITS,SHAPE_Area,NEIGHBORHO,SHAPE_Leng,longitude,latitude
0,1.0,15410.935673,Ridgedale Park,579.129308,-84.354107,33.851497
1,1.0,1343.911541,Ridgedale Park,157.003657,-84.356205,33.851182
2,1.0,19103.770553,Ridgedale Park,614.841990,-84.355119,33.852113
3,1.0,22797.417784,Ridgedale Park,721.541720,-84.349863,33.853527
4,1.0,18240.379683,Ridgedale Park,582.265904,-84.353281,33.852851
...,...,...,...,...,...,...
131887,1.0,99.999600,Oakland City,39.999920,-84.423512,33.724779
131888,1.0,99.999600,Oakland City,39.999920,-84.423470,33.724747
131889,1.0,99.999600,Atlantic Station,39.999920,-84.397755,33.790850
131890,1.0,99.999600,Atlantic Station,39.999920,-84.397755,33.790850


In [14]:
print(gdf_centroids_final.head())
print(snap_atlanta_df.head())
print(df_grocery_selected.head())
print(gdf_selected.head())

   LIVUNITS    SHAPE_Area      NEIGHBORHO  SHAPE_Leng  longitude   latitude
0       1.0  15410.935673  Ridgedale Park  579.129308 -84.354107  33.851497
1       1.0   1343.911541  Ridgedale Park  157.003657 -84.356205  33.851182
2       1.0  19103.770553  Ridgedale Park  614.841990 -84.355119  33.852113
3       1.0  22797.417784  Ridgedale Park  721.541720 -84.349863  33.853527
4       1.0  18240.379683  Ridgedale Park  582.265904 -84.353281  33.852851
                 Store Name                 Store Type     City State  \
0          Bp Garden Walk            Convenience Store  Atlanta    GA   
1                Bp To-go            Convenience Store  Atlanta    GA   
2       Chevron Food Mart            Convenience Store  Atlanta    GA   
3         Exxon Food Mart            Convenience Store  Atlanta    GA   
4  Family Dollar Store 4220  Combination Grocery/Other  Atlanta    GA   

   Latitude  Longitude  
0  33.59747  -84.42850  
1  33.62828  -84.40940  
2  33.59982  -84.42848  
3  33

In [8]:
snap_atlanta_df

,Store Name,Store Type,City,State,Latitude,Longitude
0,Bp Garden Walk,Convenience Store,Atlanta,GA,33.59747,-84.42850
1,Bp To-go,Convenience Store,Atlanta,GA,33.62828,-84.40940
2,Chevron Food Mart,Convenience Store,Atlanta,GA,33.59982,-84.42848
3,Exxon Food Mart,Convenience Store,Atlanta,GA,33.61209,-84.40353
4,Family Dollar Store 4220,Combination Grocery/Other,Atlanta,GA,33.60076,-84.42763
...,...,...,...,...,...,...
628,EL RANCHO MARKET,Convenience Store,Atlanta,GA,33.92860,-84.23786
629,Exxon Buford,Convenience Store,Atlanta,GA,33.92116,-84.25401
630,La Confianza #2,Medium Grocery Store,Atlanta,GA,33.94063,-84.26941
631,Raceway 6920 6920,Convenience Store,Atlanta,GA,33.91793,-84.25720


In [9]:
df_grocery_selected

,company_name,longitude,latitude
0,"Simply Organic Market, Inc.",-84.412667,33.718433
1,Pittsburgh Community Market,-84.399772,33.721187
2,12 Hours LLC,-84.402154,33.681607
3,"Publix Super Markets, Inc.",-84.387062,33.738851
4,MAULIA INC,-84.379616,33.709644
5,Frazie's Meat & Market,-84.471881,33.811154
6,Lee Street Meats LLC,-84.424565,33.720616
7,"HANK AARON GROCERY, LLC",-84.387655,33.732039
8,NEIGHBORHOOD 2070 INC,-84.456472,33.700385
9,Yunny LLC,-84.379880,33.754047


In [10]:
gdf_selected

,Name,Category,Longitude,Latitude
0,1055 Arden Ave. (lihtc),Garden,-84.422491,33.717346
1,A Sip of Paradise (EAV Edible Learning Garden),Community Garden,-84.344407,33.738445
2,ALDI - Buckhead,Grocery Store,-84.372448,33.840914
3,ALDI - SE Atlanta,Grocery Store,-84.350220,33.713718
4,Allen Hills aka The Commons (lihtc),Garden,-84.490200,33.752306
...,...,...,...,...
116,Whole Foods Midtown,Grocery Store,-84.388266,33.785857
117,Whole Foods Ponce,Grocery Store,-84.365988,33.774653
118,5 Points Community Garden,AgLanta Grows-A-Lot Site (legacy),-84.396702,33.749245
119,Goodr at APS Services Hub,Emergency Food Relief,-84.407371,33.752689


In [19]:
import pandas as pd


snap_std = snap_atlanta_df.rename(columns={
    "Store Name": "name",
    "Store Type": "category",
    "Longitude": "longitude",
    "Latitude": "latitude"
})[["name", "category", "longitude", "latitude"]].copy()
snap_std["source"] = "snap_atlanta_df"

grocery_std = df_grocery_selected.rename(columns={
    "company_name": "name"
})[["name", "longitude", "latitude"]].copy()
grocery_std["category"] = "Grocery Store"
grocery_std["source"] = "df_grocery_selected"

garden_std = gdf_selected.rename(columns={
    "Name": "name",
    "Category": "category",
    "Longitude": "longitude",
    "Latitude": "latitude"
})[["name", "category", "longitude", "latitude"]].copy()
garden_std["source"] = "gdf_selected"

# -----------------------------
# 3. Combine all datasets
# -----------------------------
all_stores = pd.concat([snap_std, grocery_std, garden_std], ignore_index=True)

# -----------------------------
# 4. Clean coordinates
# -----------------------------
all_stores["longitude"] = pd.to_numeric(all_stores["longitude"], errors="coerce")
all_stores["latitude"] = pd.to_numeric(all_stores["latitude"], errors="coerce")

all_stores = all_stores.dropna(subset=["longitude", "latitude"]).copy()

# round coordinates so very small decimal differences still match
all_stores["lon_round"] = all_stores["longitude"].round(5)
all_stores["lat_round"] = all_stores["latitude"].round(5)

# -----------------------------
# 5. Take union by coordinates
#    One row per unique location
# -----------------------------
union_stores = (
    all_stores.groupby(["lon_round", "lat_round"], as_index=False)
    .agg(
        longitude=("longitude", "first"),
        latitude=("latitude", "first"),
        names=("name", lambda x: " | ".join(sorted(set(x.dropna().astype(str))))),
        categories=("category", lambda x: " | ".join(sorted(set(x.dropna().astype(str))))),
        sources=("source", lambda x: " | ".join(sorted(set(x.dropna().astype(str)))))
    )
)

# -----------------------------
# 6. Optional: count how many datasets matched at each point
# -----------------------------
union_stores["n_sources"] = union_stores["sources"].apply(lambda x: len(x.split(" | ")))



In [20]:
union_stores

,lon_round,lat_round,longitude,latitude,names,categories,sources,n_sources
0,-84.61492,33.69206,-84.61492,33.69206,Clifton Station Inc,Convenience Store,snap_atlanta_df,1
1,-84.59443,33.65105,-84.59443,33.65105,2 Ray's Mart,Convenience Store,snap_atlanta_df,1
2,-84.59428,33.65076,-84.59428,33.65076,Sam's Supermarket,Convenience Store,snap_atlanta_df,1
3,-84.58299,33.72569,-84.58299,33.72569,Exxon Food Mart,Convenience Store,snap_atlanta_df,1
4,-84.57817,33.69919,-84.57817,33.69919,Publix Super Markets Inc 1718,Supermarket,snap_atlanta_df,1
...,...,...,...,...,...,...,...,...
756,-84.24462,33.90232,-84.24462,33.90232,Quik Trip 707,Convenience Store,snap_atlanta_df,1
757,-84.24346,33.88548,-84.24346,33.88548,Kroger 685,Super Store,snap_atlanta_df,1
758,-84.23904,33.89508,-84.23904,33.89508,Royal Food Mart,Convenience Store,snap_atlanta_df,1
759,-84.23786,33.92860,-84.23786,33.92860,EL RANCHO MARKET,Convenience Store,snap_atlanta_df,1


In [21]:
union_stores.to_csv("union_stores.csv", index=False)

In [ ]:
union_stores